# SetFit — fine-tuned sentence transformer

[SetFit](https://github.com/huggingface/setfit) contrastively fine-tunes the
sentence transformer (instead of freezing it like the baseline), then fits a
classification head. Same data, same seeded split, same evaluation as
`baseline_logreg.ipynb` — so the report is directly comparable.

`pip install setfit`

In [ ]:
import json
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv('labeled_demo.csv')
# CSV stores "category::label" -> keep just the label name
df['labels'] = df['labels'].apply(lambda s: [x.split('::')[-1] for x in json.loads(s)])

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['labels'])              # rows -> multi-hot label matrix
assert Y.shape == (len(df), len(mlb.classes_))
print(len(df), 'rows,', list(mlb.classes_))

In [ ]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# same split + seed as the baseline -> identical test set
text_train, text_test, Y_train, Y_test = train_test_split(
    df['text'].tolist(), Y, test_size=0.3, random_state=42)

train_ds = Dataset.from_dict({'text': text_train, 'label': Y_train.tolist()})   # multi-hot

model = SetFitModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2',
                                    multi_target_strategy='one-vs-rest')
args = TrainingArguments(batch_size=16, num_epochs=1, num_iterations=20)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

pred = np.asarray(model.predict(text_test))
print(classification_report(Y_test, pred, target_names=mlb.classes_, zero_division=0))
print('exact-match row accuracy:', round(accuracy_score(Y_test, pred), 3))

In [ ]:
results = pd.DataFrame({
    'text': text_test,
    'labels': [sorted(l) for l in mlb.inverse_transform(Y_test)],
    'predicted': [sorted(p) for p in mlb.inverse_transform(pred)],
})
results['is_correct'] = [set(t) == set(p)
                         for t, p in zip(results['labels'], results['predicted'])]
results